# Lung Sound Classification (HLS-CMDS)

Classifies **lung sound type** (Normal, Wheezing, Rhonchi, Coarse/Fine Crackles, Pleural Rub) from the [HLS-CMDS](https://doi.org/10.1109/IEEEDATA.2025.3566012) recordings, using an MLP trained on pooled log-mel spectrogram features.

## Setup

Imports: `pathlib` for file globbing, `pandas` to read the label CSVs, `librosa`/`numpy` for audio loading and feature extraction, `sklearn` for the train/test split and evaluation metrics, and `torch` for the MLP and training loop.

In [7]:
from pathlib import Path
import numpy as np, pandas as pd, librosa, torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

## Configuration

`ROOT` points at the `HLS-CMDS/` dataset folder (see `README.md` for the download/layout instructions — it's gitignored because of the `.wav` files). Audio is resampled to `SAMPLE_RATE=8000` Hz and every clip is fixed to `SECS=6.0` s so all spectrograms come out the same shape; `N_MELS`/`N_FFT`/`HOP` control the mel-spectrogram resolution.

In [8]:
ROOT = Path("HLS-CMDS")
SAMPLE_RATE = 4000
HOP_LENGTH = 256
SECS = 10.0
N_MELS = 64
LENGTH_OF_FFT = 1024

## Build the file index

Lung-labeled recordings come from two places: `LS/LS/*.wav` and the `L####.wav` files inside `Mix/Mix/` (the isolated lung component of each mixed recording, labeled by `Mix.csv`'s `Lung Sound ID`/`Lung Sound Type` columns, e.g. `L0001.wav`). Combining both sources gives 195 samples across 6 balanced classes instead of just the 50 in `LS.csv` alone.

Note: `LS.csv`'s own `Lung Sound ID` column doesn't reliably match the real filenames — it abbreviates Coarse Crackles as `G` and Fine Crackles as `C` (e.g. `M_G_LLA`), while the actual `.wav` files on disk use `CC`/`FC` (e.g. `M_CC_LLA.wav`). So for `LS/LS/` we decode the label directly from the filename's own code via a fixed `CODE2TYPE` table instead of joining on that ID column; `Mix.csv`'s numeric IDs (`L0001`, ...) don't have this problem and are looked up normally.

`CLASSES` is the sorted set of the 6 lung sound types, and `LABEL2IDX` maps each type name to the integer index used for training.

In [9]:
CODE2TYPE = {
    "N": "Normal", 
    "CC": "Coarse Crackles", 
    "FC": "Fine Crackles",
    "R": "Rhonchi", 
    "W": "Wheezing", 
    "PR": "Pleural Rub"
    }
mix_type = pd.read_csv(ROOT/"Mix.csv").set_index("Lung Sound ID")["Lung Sound Type"]

CLASSES = sorted(CODE2TYPE.values())
LABEL2IDX = {c: i for i, c in enumerate(CLASSES)}

def build_index():
    recs = [(wav, LABEL2IDX[CODE2TYPE[wav.stem.split("_")[1]]]) for wav in (ROOT/"LS"/"LS").glob("*.wav")]
    recs += [(wav, LABEL2IDX[mix_type[wav.stem]]) for wav in (ROOT/"Mix"/"Mix").glob("L*.wav")]
    return recs

## Feature extraction

`to_melspec` turns a `.wav` file into a normalized log-mel spectrogram:

1. Load audio mono at `SAMPLE_RATE` and center-crop or zero-pad it to exactly `SECS` seconds, so every clip yields a fixed-size input.
2. Compute a mel-spectrogram and convert power to decibels (`power_to_db`).
3. Standardize (zero mean, unit variance) so all clips are on a comparable scale.

An MLP takes a flat feature vector, not a 2-D image like a CNN would, and with only 195 samples a `64 x 188` spectrogram (12,032 values) would badly overfit. `to_features` pools the spectrogram over time — mean and std per mel band — collapsing it to a fixed `2 * N_MELS = 128`-dim vector that summarizes each clip's frequency content.

In [10]:
def to_melspec(path):
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    n = int(SECS*SAMPLE_RATE)
    y = np.pad(y, (0, n-len(y))) if len(y) < n else y[(len(y)-n)//2:(len(y)-n)//2+n]
    mel = librosa.feature.melspectrogram(y=y, sr=SAMPLE_RATE, n_fft=LENGTH_OF_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
    m = librosa.power_to_db(mel, ref=np.max)
    return ((m - m.mean()) / (m.std() + 1e-6)).astype(np.float32)

def to_features(path):
    spec = to_melspec(path)
    return np.concatenate([spec.mean(axis=1), spec.std(axis=1)]).astype(np.float32)

## Dataset

Thin `torch.utils.data.Dataset` wrapper: given a list of `(path, label)` pairs (from `build_index`), it lazily computes the pooled feature vector per item and returns it as a `(128,)` float tensor plus its integer class label, ready for a `DataLoader`/MLP.

In [11]:
class LungSounds(Dataset):
    def __init__(self, recs):
        self.recs = recs
    def __len__(self): return len(self.recs)
    def __getitem__(self, i):
        path, label = self.recs[i]
        return torch.from_numpy(to_features(path)), label

## Sanity check

Quick smoke test: builds the index, prints the class balance across the 6 lung sound types, and prints the feature vector's shape to confirm it's the expected fixed `128`-dim size.

In [12]:
from collections import Counter
recs = build_index()
print(Counter(CLASSES[l] for _, l in recs))   # class balance
print("feature shape:", to_features(recs[0][0]).shape)   # sanity-check pooled vector size

Counter({'Normal': 40, 'Wheezing': 35, 'Pleural Rub': 34, 'Rhonchi': 31, 'Coarse Crackles': 28, 'Fine Crackles': 27})
feature shape: (128,)


## Multi-Layer Perceptron